In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

if train.columns[0].startswith('Unnamed'):
    train = train.iloc[: , 1:]

if test.columns[0].startswith('Unnamed'):
    test = test.iloc[: , 1:]

print(f"Train shape : {train.shape}")
print(f"Test shape : {test.shape}")
print(f"\nColumns : {train.columns.tolist()[:10]}...")
print(f"\nTarget distribution : \n {train['custExit'].value_counts(normalize = True)}")

Train shape : (5424, 34)
Test shape : (1619, 33)

Columns : ['custId', 'sex', 'isElderly', 'partner', 'dependents', 'membershipDuration', 'agreementTerm', 'acquisitionChannel', 'phoneService', 'multipleLines']...

Target distribution : 
 custExit
No     0.810656
Yes    0.189344
Name: proportion, dtype: float64


In [19]:
train.head()

,custId,sex,isElderly,partner,dependents,membershipDuration,agreementTerm,acquisitionChannel,phoneService,multipleLines,...,lastContactRating,serviceSatisfactionScore,mobileAppSatisfaction,networkStabilityScore,avgNetworkLatencyMs,dataLimitWarnings,loyaltyPoints,customerFeedback,competitorOffers,custExit
0,rd84415,Male,No,Yes,No,37,One year,Referral,Yes,Yes,...,3,5,5,6.5,56,2,963,I have been a customer with this company for 3...,Yes,No
1,C60108,Male,No,No,No,39,One year,Online,Yes,No,...,Yes,3,3,8.6,38,3,934,I have been a customer with this company for a...,No,No
2,eI48924,Male,No,Yes,Yes,25,One year,Referral,Yes,No,...,5,4,2,9.5,75,2,850,I have been a customer with this company for 2...,No,No
3,G29427,Female,No,No,No,4,Month-to-month,Online,Yes,Yes,...,4,4,2,8.3,60,2,1437,I have been a customer with this company for 4...,Yes,No
4,So46165,Male,No,No,No,20,One year,Store,Yes,Yes,...,2,4,4,5.5,160,2,946,I have been a loyal customer with this interne...,No,No


In [20]:
test.head()

,custId,sex,isElderly,partner,dependents,membershipDuration,agreementTerm,acquisitionChannel,phoneService,multipleLines,...,supportTickets,lastContactRating,serviceSatisfactionScore,mobileAppSatisfaction,networkStabilityScore,avgNetworkLatencyMs,dataLimitWarnings,loyaltyPoints,customerFeedback,competitorOffers
0,m64861,Male,No,No,No,Yes,Month-to-month,Referral,Yes,No,...,Yes,5,5,Yes,3.2,148,4,615,I recently signed up for DSL internet service ...,No
1,i29953,Male,Yes,No,No,41,Month-to-month,Referral,Yes,Yes,...,No,3,5,5,3.9,44,2,1237,I recently decided to cancel my service after ...,No
2,NT11157,Male,Yes,No,No,Yes,Month-to-month,Store,Yes,No,...,3,Yes,3,Yes,4.7,153,3,594,"""I have been a customer with this fiber optic ...",No
3,c93662,Male,No,Yes,Yes,Yes,Month-to-month,Online,Yes,No,...,No,3,4,Yes,Yes,52,4,1527,I recently signed up for this service but unfo...,Yes
4,q89736,Female,No,No,No,32,One year,Store,Yes,No,...,5,Yes,4,Yes,6.8,90,4,537,I have been a loyal customer for 32 months now...,No


In [21]:
y = train['custExit'].map({'No' : 0, 'Yes' : 1})
train_ids = train['custId']
test_ids = test['custId']

X_train_full = train.drop(['custExit' , 'custId'] , axis = 1)
X_test_full = test.drop(['custId'] , axis = 1)

print(f"Features shape : {X_train_full.shape}")
print(f"Target distribution : {y.value_counts().to_dict()}")

Features shape : (5424, 32)
Target distribution : {0: 4397, 1: 1027}


In [22]:
train_feedback = X_train_full['customerFeedback'].fillna('')
test_feedback = X_test_full['customerFeedback'].fillna('')

tfidf = TfidfVectorizer(max_features = 1000, stop_words = 'english' , ngram_range = (1 , 2), min_df = 2)
train_feedback_tfidf = tfidf.fit_transform(train_feedback)
test_feedback_tfidf = tfidf.transform(test_feedback)

svd = TruncatedSVD(n_components = 100, random_state = 42)
train_feedback_svd = svd.fit_transform(train_feedback_tfidf)
test_feedback_svd = svd.transform(test_feedback_tfidf)

print(f"TF-IDF features reduced to {train_feedback_svd.shape[1]} components")
print(f"Explained variance : {svd.explained_variance_ratio_.sum():.4f}")

TF-IDF features reduced to 100 components
Explained variance : 0.5642


In [23]:
X_train_full['feedback_length'] = train_feedback.str.len()
X_test_full['feedback_length'] = test_feedback.str.len()

X_train_full['feedback_word_count'] = train_feedback.str.split().str.len()
X_test_full['feedback_word_count'] = test_feedback.str.split().str.len()

if 'competitorOffers' in X_train_full.columns:
    train_offers = X_train_full['competitorOffers'].fillna('')
    test_offers = X_test_full['competitorOffers'].fillna('')

    X_train_full['has_competitor_offer'] = (train_offers != '').astype(int)
    X_test_full['has_competitor_offer'] = (test_offers != '').astype(int)

    X_train_full['offers_length'] = train_offers.str.len()
    X_test_full['offers_length'] = test_offers.str.len()

text_cols = ['customerFeedback' , 'competitorOffers']

X_train_full = X_train_full.drop(
    columns=[col for col in text_cols if col in X_train_full.columns]
)

X_test_full = X_test_full.drop(
    columns=[col for col in text_cols if col in X_test_full.columns]
)

print(f"Created {len(['feedback_length' , 'feedback_word_count', 'has_competitor_offer', 'offers_length'])} additional text features")

Created 4 additional text features


In [24]:
train.columns

Index(['custId', 'sex', 'isElderly', 'partner', 'dependents',
       'membershipDuration', 'agreementTerm', 'acquisitionChannel',
       'phoneService', 'multipleLines', 'internetService',
       'monthlyDataUsageGb', 'cyberProtectionService', 'onlineBackup',
       'deviceProtection', 'techSupport', 'streamingTv', 'streamingMovies',
       'recurringFee', 'cumulativeSpend', 'transactionMethod',
       'paperlessBilling', 'billingIssues', 'supportTickets',
       'lastContactRating', 'serviceSatisfactionScore',
       'mobileAppSatisfaction', 'networkStabilityScore', 'avgNetworkLatencyMs',
       'dataLimitWarnings', 'loyaltyPoints', 'customerFeedback',
       'competitorOffers', 'custExit'],
      dtype='object')

In [25]:
numeric_column_names = ['membershipDuration', 'monthlyDataUsageGb' , 'recurringFee', 'cumulativeSpend',
                   'supportTickets', 'lastContactRating', 'serviceSatisfactionScore', 'networkStabilityScore', 
                   'avgNetworkLatencyMs', 'loyaltyPoints', 'billingIssues', 'dataLimitWarnings', 'mobileAppSatisfaction']

for col in numeric_column_names : 
    if col in X_train_full.columns:
        X_train_full[col] = pd.to_numeric(X_train_full[col], errors = 'coerce')
        X_test_full[col] = pd.to_numeric(X_test_full[col], errors = 'coerce')

print("Numeric columns converted successfully")


Numeric columns converted successfully


In [26]:
X_train_full['spend_per_month'] = X_train_full['cumulativeSpend'] / (X_train_full['membershipDuration'] + 1)
X_test_full['spend_per_month'] = X_test_full['cumulativeSpend'] / (X_test_full['membershipDuration'] + 1)

X_train_full['issues_per_month'] = X_train_full['billingIssues'] / (X_train_full['membershipDuration'] + 1)
X_test_full['issues_per_month'] = X_test_full['billingIssues'] / (X_test_full['membershipDuration'] + 1)

X_train_full['tickets_per_month'] = X_train_full['supportTickets'] / (X_train_full['membershipDuration'] + 1)
X_test_full['tickets_per_month'] = X_test_full['supportTickets'] / (X_test_full['membershipDuration'] + 1)

X_train_full['loyalty_per_month'] = X_train_full['loyaltyPoints'] / (X_train_full['membershipDuration'] + 1)
X_test_full['loyalty_per_month'] = X_test_full['loyaltyPoints'] / (X_test_full['membershipDuration'] + 1)

satisfaction_cols = ['lastContactRating' , 'serviceSatisfactionScore']

X_train_full['avg_satisfaction'] = X_train_full[satisfaction_cols].mean(axis=1)
X_test_full['avg_satisfaction'] = X_test_full[satisfaction_cols].mean(axis=1)

print("Created 5 engineered features")


Created 5 engineered features


In [27]:
categorical_cols = X_train_full.select_dtypes(include = ['object'])
print(f"Categorical columns : {categorical_cols}")

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train_full[col] = X_train_full[col].fillna('Missing')
    X_test_full[col] = X_test_full[col].fillna('Missing')

    le.fit(pd.concat([X_train_full[col] , X_test_full[col]]))
    X_train_full[col] = le.transform(X_train_full[col])
    X_test_full[col] = le.transform(X_test_full[col])
    le_dict[col] = le

print(f"Encoded {len(categorical_cols)} categorical columns")

Categorical columns :          sex isElderly partner dependents   agreementTerm acquisitionChannel  \
0       Male        No     Yes         No        One year           Referral   
1       Male        No      No         No        One year             Online   
2       Male        No     Yes        Yes        One year           Referral   
3     Female        No      No         No  Month-to-month             Online   
4       Male        No      No         No        One year              Store   
...      ...       ...     ...        ...             ...                ...   
5419    Male        No      No         No  Month-to-month              Agent   
5420  Female        No     Yes        Yes  Month-to-month              Store   
5421    Male        No      No         No  Month-to-month              Store   
5422    Male        No     Yes         No        Two year             Online   
5423    Male       Yes      No         No  Month-to-month             Online   

     phoneService

In [28]:
numeric_cols = X_train_full.select_dtypes(include = [np.number]).columns
missing_count = 0
for col in numeric_cols:
    if X_train_full[col].isnull().sum() > 0:
        median_val = X_train_full[col].median()
        X_train_full[col].fillna(median_val, inplace = True)
        X_test_full[col].fillna(median_val, inplace = True)

        missing_count += 1
print(f"Imputed missing values in {missing_count} columns")

Imputed missing values in 13 columns


In [30]:
feedback_cols = [
    f'feedback_svd_{i}'
    for i in range(train_feedback_svd.shape[1])
]
train_feed_back_df = pd.DataFrame(train_feedback_svd, columns = feedback_cols)
test_feed_back_df = pd.DataFrame(test_feedback_svd, columns = feedback_cols)

X_train_full = pd.concat([X_train_full, train_feed_back_df], axis = 1)
X_test_full = pd.concat([X_test_full, test_feed_back_df], axis = 1)

print(f"\nFinal feature shape - Train : {X_train_full.shape} , Test : {X_test_full.shape}")


Final feature shape - Train : (5424, 139) , Test : (1619, 139)


In [31]:
X_train, X_val , y_train, y_val = train_test_split(X_train_full, y, test_size = 0.2, random_state = 42, stratify = y)
print(f"Train : {X_train.shape} , Validation : {X_val.shape}")

Train : (4339, 139) , Validation : (1085, 139)


In [33]:
models = {}

print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators = 300,
    max_depth = 6,
    learning_rate = 0.5,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42,
    eval_metric = 'auc',
    use_Label_encoder = False)

xgb.fit(X_train, y_train)
models['xgb'] = xgb
print("XGBoost trained")

Training XGBoost...


[14:59:14] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_Label_encoder" } are not used.



XGBoost trained


In [34]:
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(
    n_estimators = 200,
    max_depth = 5,
    learning_rate = 0.5,
    subsample = 0.8,
    random_state = 42)

gb.fit(X_train, y_train)
models['gb'] = gb
print("Gradient Boosting trained")

Training Gradient Boosting...
Gradient Boosting trained


In [35]:
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators = 200,
    max_depth = 10,
    min_samples_split = 10,
    min_samples_leaf = 4,
    random_state = 4,
    n_jobs = -1)

rf.fit(X_train, y_train)
models['rf'] = rf
print("Random Forest trained")

Training Random Forest...
Random Forest trained


In [37]:
print("Training Logistic Regression...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr = LogisticRegression(max_iter = 1000, C = 0.1, random_state = 42)
lr.fit(X_train_scaled, y_train)
models['lr'] = lr
print("Logistic Regression trained")


Training Logistic Regression...
Logistic Regression trained


In [38]:
val_predictions = {}
for name, model in models.items():
    if name == 'lr':
        y_pred = model.predict_proba(X_val_scaled)[: , 1]
    else:
        y_pred = model.predict_proba(X_val)[: , 1]

    val_predictions[name] = y_pred
    auc_score = roc_auc_score(y_val , y_pred)

    print(f"{name.upper():15s} : AUC-ROC = {auc_score:.4f} (Score : {auc_score * 100:.4f})")

XGB             : AUC-ROC = 1.0000 (Score : 99.9961)
GB              : AUC-ROC = 1.0000 (Score : 99.9967)
RF              : AUC-ROC = 1.0000 (Score : 99.9978)
LR              : AUC-ROC = 1.0000 (Score : 100.0000)


In [39]:
weights = {
    'xgb' : 1,
    'gb' : 0,
    'rf' : 1,
    'lr' : 1
}

ensemble_pred = np.zeros(len(y_val))
for name, weight in weights.items():
    ensemble_pred += weight * val_predictions[name]

ensemble_pred /= sum(weights.values())

ensemble_auc = roc_auc_score(y_val, ensemble_pred)
print(f"{'ENSEMBLE':15s}: AUC-ROC : {ensemble_auc:.4f} (Score : {ensemble_auc * 100:.4f})")

ENSEMBLE       : AUC-ROC : 1.0000 (Score : 100.0000)


In [41]:
# TODO:
test_predictions = {}
for name , model in models.items():
    if name == 'lr':
        X_test_scaled = scaler.transform(X_test_full)
        y_pred = model.predict_proba(X_test_scaled)[: , 1]

    else:
        y_pred = model.predict_proba(X_test_full)[: , 1]

    test_predictions[name] = y_pred
    print(f"{name.upper()} : min : {y_pred.min():.4f} , max : {y_pred.max():.4f} , mean : {y_pred.mean():.4f} ")

test_ensemble_pred = np.zeros(len(X_test_full))
for name, weight in weights.items():
    test_ensemble_pred += weight * test_predictions[name]

test_ensemble_pred  /= sum(weights.values())

print(f"{'ENSEMBLE':15s}: min : {test_ensemble_pred.min():.4f} , max : {test_ensemble_pred.max():.4f} , mean : {test_ensemble_pred.mean():.4f}")

XGB : min : 0.0000 , max : 1.0000 , mean : 0.5047 
GB : min : 0.0000 , max : 1.0000 , mean : 0.5065 
RF : min : 0.0011 , max : 0.9878 , mean : 0.4298 
LR : min : 0.0000 , max : 1.0000 , mean : 0.5075 
ENSEMBLE       : min : 0.0007 , max : 0.9955 , mean : 0.4807


In [45]:
submission = pd.DataFrame({
    'custId' : test_ids.values,
    'custExit' : test_ensemble_pred 
})

submission.head(10)

,custId,custExit
0,m64861,0.956123
1,i29953,0.958335
2,NT11157,0.022209
3,c93662,0.718017
4,q89736,0.006528
5,x17800,0.020865
6,K25203,0.025138
7,J84525,0.913204
8,k14074,0.008359
9,e63909,0.986594
